### Process data

In [1]:
### Read the fasta file
from Bio import SeqIO
import gzip
import pandas as pd

fasta_path = "/scratch/st-cdeboer-1/sambina/position_mpra/data/GTeX_experimental_data/ENCFF728XQT.fasta.gz"

records = list(SeqIO.parse(gzip.open(fasta_path, "rt"), "fasta"))
print(f"Loaded {len(records)} sequences")
print(records[0].id)

fasta_df = pd.DataFrame({
    "id": [rec.id for rec in records],
    "sequence": [str(rec.seq) for rec in records]
})

fasta_df["base_id"] = fasta_df["id"].str.extract(r"^([^:]+:[^:]+)")
print(fasta_df.head())

Loaded 243698 sequences
10:100014847:CT:C:A:wC
                       id                                           sequence  \
0  10:100014847:CT:C:A:wC  TTGGCACTGACCTGGCTGGGAGATGTCAGAGATGCTGCTGCCCTAA...   
1  10:100014847:CT:C:R:wC  TTGGCACTGACCTGGCTGGGAGATGTCAGAGATGCTGCTGCCCTAA...   
2   10:100025816:G:A:A:wC  TCTGACCCCTCCCACGCTTCCTTTCCAGACTGAAACACTGTCTTCT...   
3   10:100025816:G:A:R:wC  TCTGACCCCTCCCACGCTTCCTTTCCAGACTGAAACACTGTCTTCT...   
4   10:100029561:C:T:A:wC  GCAACACAGAGGGTTCGGAAACGTGGGCTCTGCCTGCTGCCAATCA...   

        base_id  
0  10:100014847  
1  10:100014847  
2  10:100025816  
3  10:100025816  
4  10:100029561  


In [2]:
### Read the csv file
data_var_effects = pd.read_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/GTeX_experiment/42_variants.csv")

In [3]:
# Merge fasta_df with data_var_effects
merged_df = fasta_df.merge(
    data_var_effects,
    left_on="base_id",   # from fasta_df
    right_on="base_id"
)

print(merged_df.head())
print(f"Merged shape: {merged_df.shape}")


                     id                                           sequence  \
0  10:35356774:A:G:A:wC  AAAAAAAAAAAATTCACCATTATTGTTTATTCTATTTATATAAAGT...   
1  10:35356774:A:G:A:wL  GAAAATTCACCATTTAAGAACCACAAAAGGGCCGGACGTGGTGGCT...   
2  10:35356774:A:G:A:wR  TCACACCACTGCACTCCAACCTGGGCAACAGAGTGAGACCCTGCCA...   
3  10:35356774:A:G:R:wC  AAAAAAAAAAAATTCACCATTATTGTTTATTCTATTTATATAAAGT...   
4  10:35356774:A:G:R:wL  GAAAATTCACCATTTAAGAACCACAAAAGGGCCGGACGTGGTGGCT...   

       base_id  Unnamed: 0    centre      left     right  
0  10:35356774           0 -0.693531 -1.012101 -0.237772  
1  10:35356774           0 -0.693531 -1.012101 -0.237772  
2  10:35356774           0 -0.693531 -1.012101 -0.237772  
3  10:35356774           0 -0.693531 -1.012101 -0.237772  
4  10:35356774           0 -0.693531 -1.012101 -0.237772  
Merged shape: (258, 7)


### Load Model

In [4]:
import pandas as pd
import torch
from torchinfo import summary
from tqdm import tqdm
import numpy as np
import sys
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
import json
import os
sys.path.append("/scratch/st-cdeboer-1/sambina/mpra/models/random-promoter-dream-challenge-2022/benchmarks/human")

from prixfixe.autosome import AutosomeFinalLayersBlock
from prixfixe.bhi import BHIFirstLayersBlock, BHICoreBlock
from prixfixe.prixfixe import PrixFixeNet

def configure_device(device):
    """Configures the CUDA device."""
    device = torch.device(device)
    generator = torch.Generator()
    generator.manual_seed(42)
    return device, generator

def initialize_model(seq_size, generator):
    """Initializes the PrixFixeNet model."""
    first = BHIFirstLayersBlock(
        in_channels=5,
        out_channels=320,
        seqsize=seq_size,
        kernel_sizes=[9, 15],
        pool_size=1,
        dropout=0.2
    )

    core = BHICoreBlock(
        in_channels=first.out_channels,
        out_channels=320,
        seqsize=first.infer_outseqsize(),
        lstm_hidden_channels=320,
        kernel_sizes=[9, 15],
        pool_size=1,
        dropout1=0.2,
        dropout2=0.5
    )

    final = AutosomeFinalLayersBlock(in_channels=core.out_channels)

    model = PrixFixeNet(
        first=first,
        core=core,
        final=final,
        generator=generator
    )

    return model

def load_model_weights(model, model_log_dir, device):
    """Loads model weights from the specified path."""
    print(device)
    model.load_state_dict(torch.load(model_log_dir, map_location=torch.device(device)))
    model.eval()
    return model

def print_model_summary(model, seq_size):
    """Prints the model summary."""
    print(summary(model, (1, 5, seq_size)))
    
def add_reverse_column(filtered_data):
    """Add a reverse column indicating if '_Reversed:' is present in the Sequence_ID."""
    filtered_data['rev'] = filtered_data['id'].str.contains('_Reversed:').astype(int)
    return filtered_data


def one_hot_encode(seq):
    """One-hot encode a DNA sequence."""
    mapping = {
        'A': [1, 0, 0, 0],
        'G': [0, 1, 0, 0],
        'C': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'N': [0, 0, 0, 0]
    }
    return [mapping[base] for base in seq]


def encode_sequences(test_df, model_rnn, SEQ_SIZE, device):
    """One-hot encode the sequences and run predictions using the RNN model.
    If sequence length is < 200, add 'N' at the rightmost side.
    """
    encoded_seqs = []
    pred_expr_rnn = []
    for i, row in tqdm(test_df.iterrows()):
        seq = row['sequence']
        if len(seq) < 200:
            seq = 'N'*(200-len(seq)) + seq 
        seq = "ACTGGCCGCTTGACG" + seq + "CACTGCGGCTCCTGCG"

        encoded_seq = one_hot_encode(seq)
        rev_value = [row['rev']] * len(encoded_seq)
        encoded_seq_with_rev = [list(encoded_base) + [rev] for encoded_base, rev in zip(encoded_seq, rev_value)]
        
        pred = model_rnn(torch.tensor(np.array(encoded_seq_with_rev).reshape(1, SEQ_SIZE, 5).transpose(0, 2, 1), device=device, dtype=torch.float32))
        pred_expr_rnn.append(pred.detach().cpu().flatten().tolist())
    
    return pred_expr_rnn


def save_predictions(test_df, pred_expr_rnn, OUTPUT):
    """Save the predicted values to a file."""
    os.makedirs(OUTPUT, exist_ok=True)

    file_path = f"{OUTPUT}/predicted_mean.txt"
    df = pd.DataFrame({
        'seq_id': test_df['id'],
        'prediction': pred_expr_rnn
    })
    
    df['prediction'] = df['prediction'].apply(lambda x: x[0])
    df.to_csv(file_path, index=False)
    return df


def main():
    CUDA_DEVICE_ID = "cpu"
    OUTPUT = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/GTeX_experiment/"
    MODEL_LOG_DIR = "/scratch/st-cdeboer-1/sambina/mpra/output/chromosome/gosai/output_lfcse/output_k562/fold_4/model_best.pth"
    sys.path.append("/scratch/st-cdeboer-1/sambina/mpra/models/random-promoter-dream-challenge-2022/benchmarks/human")

    SEQ_SIZE = 231

    device, generator = configure_device(CUDA_DEVICE_ID)
    model_rnn = initialize_model(SEQ_SIZE, generator)
    print_model_summary(model_rnn, SEQ_SIZE)
    model_rnn = load_model_weights(model_rnn, MODEL_LOG_DIR, device)


    filtered_data_reversed = add_reverse_column(merged_df)
    pred_expr_rnn = encode_sequences(filtered_data_reversed, model_rnn, SEQ_SIZE, device)
    predict_df = save_predictions(filtered_data_reversed, pred_expr_rnn, OUTPUT)

if __name__ == "__main__":
    main()


/arc/project/st-cdeboer-1/sambina/miniconda3/envs/dream_rocky/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Layer (type:depth-idx)                   Output Shape              Param #
PrixFixeNet                              [1, 1]                    --
├─BHIFirstLayersBlock: 1-1               --                        --
│    └─ModuleList: 2-1                   --                        --
│    │    └─ConvBlock: 3-1               [1, 160, 231]             7,360
│    │    └─ConvBlock: 3-2               [1, 160, 231]             12,160
├─BHICoreBlock: 1-2                      --                        --
│    └─LSTM: 2-2                         [1, 231, 640]             1,643,520
│    └─ModuleList: 2-3                   --                        --
│    │    └─ConvBlock: 3-3               [1, 160, 231]             921,760
│    │    └─ConvBlock: 3-4               [1, 160, 231]             1,536,160
│    └─Dropout: 2-4                      [1, 320, 231]             --
├─AutosomeFinalLayersBlock: 1-3          --                        --
│    └─Conv1d: 2-5                       [1, 256, 231]     

258it [02:59,  1.44it/s]


### Analyse predicted positional effects

In [7]:
predictions = pd.read_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/GTeX_experiment/predicted_mean.txt")

parts = predictions["seq_id"].str.split(":", expand=True)

# Assign new columns
predictions["base_id"] = parts[0] + ":" + parts[1]
predictions["allele"] = parts[4]
predictions["window"] = parts[5]
predictions

,seq_id,prediction,base_id,allele,window
0,10:35356774:A:G:A:wC,1.311615,10:35356774,A,wC
1,10:35356774:A:G:A:wL,1.738686,10:35356774,A,wL
2,10:35356774:A:G:A:wR,0.145187,10:35356774,A,wR
3,10:35356774:A:G:R:wC,1.034276,10:35356774,R,wC
4,10:35356774:A:G:R:wL,1.187440,10:35356774,R,wL
...,...,...,...,...,...
253,9:131038575:G:C:A:wL,3.862663,9:131038575,A,wL
254,9:131038575:G:C:A:wR,7.670659,9:131038575,A,wR
255,9:131038575:G:C:R:wC,7.004169,9:131038575,R,wC
256,9:131038575:G:C:R:wL,3.509566,9:131038575,R,wL
